# カウンティング法（選択率の単純集計）

カウンティング法（選択率の単純集計）による効用値・属性重要度の推定。

水準ごとの選択率（選択回数 ÷ 提示回数）の対数を効用の代用値とみなす簡易法。
実データ（`camera_long.csv`）・疑似データ（`persearch_camera_long.csv`）のどちらにも
同じ関数を適用し、同一の long スキーマ（1行 = 1課題 × 1選択肢）から集計する。

理論的な弱点（同一選択セット内でしか `ln P_i − ln P_j = V_i − V_j` が成り立たない
こと）は Qiita 記事側で説明する。ここではその弱点込みで「手早く算出できる簡易法」
として実装する。

集計ロジックは `conjoint_lib.counting` にまとまっている（`run_counting.py` からも
同じ関数を呼んでいる）。


In [1]:
from __future__ import annotations

import inspect

from conjoint_lib import schema
from conjoint_lib.counting import attribute_importance, counting_utilities, run_counting_all


## 属性グループ

- `MAIN_ATTRS`: 5属性モデル（persearch の疑似データと直接比較する主分析）
- `FULL_ATTRS`: 全属性モデル（video・swivel を含む頑健性チェック）


In [2]:
print("MAIN_ATTRS:", schema.MAIN_ATTRS)
print("FULL_ATTRS:", schema.FULL_ATTRS)


MAIN_ATTRS: ['brand', 'pixels', 'zoom', 'wifi']
FULL_ATTRS: ['brand', 'pixels', 'zoom', 'video', 'swivel', 'wifi']


## アルゴリズム

記事の主題そのものなので、`conjoint_lib.counting` の実装をそのまま表示する。

In [3]:
print(inspect.getsource(counting_utilities))


def counting_utilities(
    df: pd.DataFrame,
    attr_cols: list[str],
    price_col: str | None = None,
) -> pd.DataFrame:
    """属性×水準ごとの提示回数・選択回数・選択率・効用値(代用値)を計算する。

    Args:
        df: long 形式 DataFrame（1行=1課題×1選択肢、is_none/chosen 列を持つ）
        attr_cols: 集計するカテゴリ属性の列名（price を除く）
        price_col: 価格属性の列名（数値）。None なら価格は集計しない

    Returns:
        columns: attr, level, shown, chosen, rate, log_rate, utility(基準水準=0),
                 is_reference
    """
    shown_df = df[df["is_none"] == 0]

    rows: list[dict] = []
    all_attrs = list(attr_cols)
    level_orders: dict[str, list] = {a: schema.ATTRIBUTE_LEVEL_ORDER[a] for a in attr_cols}
    if price_col is not None:
        all_attrs = all_attrs + [price_col]
        level_orders[price_col] = _price_level_order(df, price_col)

    for attr in all_attrs:
        levels = level_orders[attr]
        for level_idx, level in enumerate(levels):
            mask = shown_df[attr] == level
            shown = int(mask.sum())
           

In [4]:
print(inspect.getsource(attribute_importance))


def attribute_importance(counting_result: pd.DataFrame) -> pd.DataFrame:
    """属性重要度（属性内 utility の幅を全属性の幅の合計で正規化）を計算する。"""
    width = counting_result.groupby("attr")["utility"].agg(lambda s: s.max() - s.min())
    importance = (width / width.sum()).sort_values(ascending=False)
    return importance.rename("importance").reset_index()



## 実行

`prepare_camera.ipynb` / `prepare_persearch_camera.ipynb` を先に実行しておくこと。

In [5]:
run_counting_all()


=== camera / main5 ===
        attr     level  shown  chosen     rate  log_rate  is_reference   utility
       brand     canon   4948    1108 0.223929 -1.496427          True  0.000000
       brand      sony   5366     975 0.181700 -1.705401         False -0.208974
       brand     nikon   5454    1058 0.193986 -1.639969         False -0.143542
       brand panasonic   5480     828 0.151095 -1.889847         False -0.393420
      pixels       低画素  10557    1509 0.142938 -1.945342          True  0.000000
      pixels       高画素  10691    2460 0.230100 -1.469241         False  0.476101
        zoom     標準ズーム  10394    1482 0.142582 -1.947836          True  0.000000
        zoom    高倍率ズーム  10854    2487 0.229132 -1.473457         False  0.474380
        wifi   Wi-Fiなし  10637    1601 0.150512 -1.893710          True  0.000000
        wifi   Wi-Fiあり  10611    2368 0.223165 -1.499845         False  0.393865
price_usd100      0.79   4057    1853 0.456741 -0.783638          True  0.000000
price

=== persearch_camera / main5 ===


        attr     level  shown  chosen     rate  log_rate  is_reference   utility
       brand     canon  10624    1657 0.155968 -1.858107          True  0.000000
       brand      sony  10624    1846 0.173758 -1.750094         False  0.108012
       brand     nikon  10624    2026 0.190700 -1.657052         False  0.201055
       brand panasonic  10624    2724 0.256401 -1.361014         False  0.497093
      pixels       低画素  21248    3172 0.149285 -1.901900          True  0.000000
      pixels       高画素  21248    5081 0.239128 -1.430755         False  0.471146
        zoom     標準ズーム  21248    3674 0.172910 -1.754982          True  0.000000
        zoom    高倍率ズーム  21248    4579 0.215503 -1.534782         False  0.220200
        wifi   Wi-Fiなし  21248    2029 0.095491 -2.348720          True  0.000000
        wifi   Wi-Fiあり  21248    6224 0.292922 -1.227850         False  1.120870
price_usd100      0.79   8632    4039 0.467910 -0.759479          True  0.000000
price_usd100      1.29   83

[WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_camera_main5.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_camera_main5_importance.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_camera_full.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_camera_full_importance.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_persearch_camera_main5.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_persearch_camera_main5_importance.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/counting_persearch_camera_full.csv'),
 WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/c